In [3]:
# Install the Ultralytics library, which includes the YOLO (You Only Look Once) models and tools for object detection and pose estimation
!pip install ultralytics

# Install OpenCV, a computer vision library used for image and video processing
!pip install opencv-python

In [4]:
# Deep Learning / Object Detection
from ultralytics import YOLO  # YOLO model for object detection and pose estimation

# Computer Vision
import cv2  # OpenCV for image and video processing
import time

# Data Handling
import pandas as pd  # For working with dataframes (structured data)
import numpy as np   # For numerical computations and array operations
import os            # For interacting with the operating system (e.g., file paths)
from glob import glob  # For finding file paths matching a pattern
from glob import iglob # For finding file paths matching a pattern

# Machine Learning
import joblib        # For saving and loading machine learning models

# Machine Learning
from xgboost import XGBClassifier  # Gradient boosting classifier
from sklearn.model_selection import train_test_split  # To split data into training and test sets
from sklearn.metrics import classification_report      # To evaluate classification model performance

In [5]:
import os

# Change this to wherever you want to store your data
base_dir = r'C:\Users\derek\.cache\kagglehub\datasets\hasyimabdillah\workoutfitness-video\versions\5\push-up' # Windows
# base_dir = "/home/yourname/exercise_form"              # Mac/Linux

paths = {
    "good_form_videos":    os.path.join(base_dir, 'good_form'),
    "bad_form_videos":     os.path.join(base_dir, 'bad_form'),
    "good_form_keypoints": os.path.join(base_dir, 'exercise_keypoints', 'good_form'),
    "bad_form_keypoints":  os.path.join(base_dir, 'exercise_keypoints', 'bad_form'),
    "good_form_labeled":   os.path.join(base_dir, 'exercise_labeled_videos', 'good_form'),
    "bad_form_labeled":    os.path.join(base_dir, 'exercise_labeled_videos', 'bad_form'),
    "test_videos":         os.path.join(base_dir, 'test_videos'),
}

for path_name, path in paths.items():
    os.makedirs(path, exist_ok=True)
    print(f"Checked/created: {path_name} -> {path}")

Checked/created: good_form_videos -> C:\Users\derek\.cache\kagglehub\datasets\hasyimabdillah\workoutfitness-video\versions\5\push-up\good_form
Checked/created: bad_form_videos -> C:\Users\derek\.cache\kagglehub\datasets\hasyimabdillah\workoutfitness-video\versions\5\push-up\bad_form
Checked/created: good_form_keypoints -> C:\Users\derek\.cache\kagglehub\datasets\hasyimabdillah\workoutfitness-video\versions\5\push-up\exercise_keypoints\good_form
Checked/created: bad_form_keypoints -> C:\Users\derek\.cache\kagglehub\datasets\hasyimabdillah\workoutfitness-video\versions\5\push-up\exercise_keypoints\bad_form
Checked/created: good_form_labeled -> C:\Users\derek\.cache\kagglehub\datasets\hasyimabdillah\workoutfitness-video\versions\5\push-up\exercise_labeled_videos\good_form
Checked/created: bad_form_labeled -> C:\Users\derek\.cache\kagglehub\datasets\hasyimabdillah\workoutfitness-video\versions\5\push-up\exercise_labeled_videos\bad_form
Checked/created: test_videos -> C:\Users\derek\.cache\

In [6]:
good_form_path = r"C:\Users\derek\.cache\kagglehub\datasets\hasyimabdillah\workoutfitness-video\versions\5\push-up\good_form"
bad_form_path = r"C:\Users\derek\.cache\kagglehub\datasets\hasyimabdillah\workoutfitness-video\versions\5\push-up\bad_form"

good_form_keypoints_path = r"C:\Users\derek\.cache\kagglehub\datasets\hasyimabdillah\workoutfitness-video\versions\5\push-up\exercise_keypoints\good_form"
bad_form_keypoints_path = r"C:\Users\derek\.cache\kagglehub\datasets\hasyimabdillah\workoutfitness-video\versions\5\push-up\exercise_keypoints\bad_form"

good_form_labeled_video_path = r"C:\Users\derek\.cache\kagglehub\datasets\hasyimabdillah\workoutfitness-video\versions\5\push-up\exercise_labeled_videos\good_form"
bad_form_labeled_video_path = r"C:\Users\derek\.cache\kagglehub\datasets\hasyimabdillah\workoutfitness-video\versions\5\push-up\exercise_labeled_videos\bad_form"

test_videos_path = r"C:\Users\derek\.cache\kagglehub\datasets\hasyimabdillah\workoutfitness-video\versions\5\push-up\test_videos"

print("All paths defined successfully!")
print("Good form videos:", good_form_path)
print("Bad form videos:", bad_form_path)

All paths defined successfully!
Good form videos: C:\Users\derek\.cache\kagglehub\datasets\hasyimabdillah\workoutfitness-video\versions\5\push-up\good_form
Bad form videos: C:\Users\derek\.cache\kagglehub\datasets\hasyimabdillah\workoutfitness-video\versions\5\push-up\bad_form


In [21]:
# --- Load YOLO Pose Model ---
model = YOLO("yolov8n-pose.pt")  # Load the YOLOv8 nano pose model (alternatives: yolov8s/m for better accuracy)

# Loop through good and bad form folders with label 0 (good) and 1 (bad)
for label, folder in enumerate([good_form_path, bad_form_path]):
    video_files = []
    for ext in ["*.mp4", "*.MP4", "*.Mp4", "*.mP4", ".mp4"]:
        video_files.extend(iglob(os.path.join(folder, ext)))
    # Get all .MP4 files in the current folder
    for video_path in video_files:
        short_video_path = os.path.basename(video_path)  # Extract just the video filename
        print(short_video_path)

        # Define output paths for labeled video and keypoints CSV
        if folder == good_form_path:
            output_video_path = os.path.join(good_form_labeled_video_path, "output_" + short_video_path)
            output_csv = os.path.join(good_form_keypoints_path, short_video_path.replace(".MP4", "_keypoints.csv"))
        elif folder == bad_form_path:
            output_video_path = os.path.join(bad_form_labeled_video_path, "output_" + short_video_path)
            output_csv = os.path.join(bad_form_keypoints_path, short_video_path.replace(".MP4", "_keypoints.csv"))

        # --- Initialize Video Capture ---
        cap = cv2.VideoCapture(video_path)  # Open the video file
        frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))   # Get video width
        frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) # Get video height
        fps = cap.get(cv2.CAP_PROP_FPS)                        # Get frames per second

        # --- Initialize Video Writer for Annotated Output ---
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Define the codec
        out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))  # Set up the writer

        frame_num = 0  # Frame counter
        data = []      # List to store keypoints data for CSV
        frame_skip = 1  # Set frame interval (1 = every frame, 2 = every other frame, etc.)

        # --- Process Each Frame in the Video ---
        while cap.isOpened():
            ret, frame = cap.read()  # Read next frame
            if not ret:
                break  # Exit loop if no more frames

            # Only process selected frames (e.g., every 1st, 3rd, etc.)
            if frame_num % frame_skip == 0:
                results = model(frame, verbose=False)  # Run YOLO pose estimation

                # Draw keypoints and skeletons on the frame
                annotated_frame = results[0].plot()

                keypoints = results[0].keypoints  # Get keypoint data from results
                if keypoints is not None and len(keypoints.xy) > 0:
                    for i, kp in enumerate(keypoints.xy[0]):  # Loop through keypoints for the first detected person
                        x, y = kp[0].item(), kp[1].item()        # Extract x, y coordinates
                        conf = keypoints.conf[0][i].item()       # Confidence score
                        data.append({
                            'frame': frame_num,
                            'keypoint_id': i,
                            'x': x,
                            'y': y,
                            'confidence': conf
                        })

                # Write the annotated frame to the output video
                out.write(annotated_frame)

            frame_num += 1  # Increment frame count

        # --- Finalize Video and Save Keypoints ---
        cap.release()  # Release video capture
        out.release()  # Release video writer

        # Save keypoints data to a CSV file
        df = pd.DataFrame(data)
        df.to_csv(output_csv, index=False)

        # Print confirmation messages
        print(f"Keypoints saved to {output_csv}")
        print(f"Video saved to {output_video_path}")
        print("")

push-up_13.mp4
Keypoints saved to C:\Users\derek\.cache\kagglehub\datasets\hasyimabdillah\workoutfitness-video\versions\5\push-up\exercise_keypoints\good_form\push-up_13.mp4
Video saved to C:\Users\derek\.cache\kagglehub\datasets\hasyimabdillah\workoutfitness-video\versions\5\push-up\exercise_labeled_videos\good_form\output_push-up_13.mp4

push-up_20.mp4
Keypoints saved to C:\Users\derek\.cache\kagglehub\datasets\hasyimabdillah\workoutfitness-video\versions\5\push-up\exercise_keypoints\good_form\push-up_20.mp4
Video saved to C:\Users\derek\.cache\kagglehub\datasets\hasyimabdillah\workoutfitness-video\versions\5\push-up\exercise_labeled_videos\good_form\output_push-up_20.mp4

push-up_37.mp4
Keypoints saved to C:\Users\derek\.cache\kagglehub\datasets\hasyimabdillah\workoutfitness-video\versions\5\push-up\exercise_keypoints\good_form\push-up_37.mp4
Video saved to C:\Users\derek\.cache\kagglehub\datasets\hasyimabdillah\workoutfitness-video\versions\5\push-up\exercise_labeled_videos\good_fo

In [7]:
# --- Helper Function to Extract Features from a Keypoints CSV File ---
def extract_features_from_csv(csv_path):
    # Load keypoints CSV into a DataFrame
    df = pd.read_csv(csv_path)

    # Pivot the data: rows = frames, columns = (x, y) coordinates for each keypoint_id
    grouped = df.groupby(['frame', 'keypoint_id'])[['x', 'y']].mean().unstack()

    # --- Feature Extraction ---
    # Use statistical summaries:
    features = grouped.mean().tolist()  # Mean position of each keypoint (avg posture)
    features += grouped.std().tolist()  # Standard deviation (movement/instability)

    return features

# --- Load All Keypoint Data and Labels ---
X = []  # Feature vectors for each video
y = []  # Corresponding labels: 0 = good form, 1 = bad form

# Loop through both good and bad form keypoints folders
for label, folder in enumerate([good_form_keypoints_path, bad_form_keypoints_path]):
    matches = glob(os.path.join(folder, "*.mp4"))
    print(f"Found {len(matches)} CSVs in: {folder}")
    for path in matches:
        features = extract_features_from_csv(path)
        if features:
            X.append(features)
            y.append(label)

# Convert to arrays
X = np.array(X)
y = np.array(y)

Found 8 CSVs in: C:\Users\derek\.cache\kagglehub\datasets\hasyimabdillah\workoutfitness-video\versions\5\push-up\exercise_keypoints\good_form
Found 48 CSVs in: C:\Users\derek\.cache\kagglehub\datasets\hasyimabdillah\workoutfitness-video\versions\5\push-up\exercise_keypoints\bad_form


In [8]:
import os
print(os.listdir(good_form_keypoints_path))
print(os.listdir(bad_form_keypoints_path))

['push-up_13.mp4', 'push-up_20.mp4', 'push-up_37.mp4', 'push-up_40.mp4', 'push-up_43.mp4', 'push-up_54.mp4', 'push-up_55.mp4', 'push-up_56.mp4']
['push-up_1.mp4', 'push-up_10.mp4', 'push-up_11.mp4', 'push-up_12.mp4', 'push-up_14.mp4', 'push-up_15.mp4', 'push-up_16.mp4', 'push-up_17.mp4', 'push-up_18.mp4', 'push-up_19.mp4', 'push-up_2.mp4', 'push-up_21.mp4', 'push-up_22.mp4', 'push-up_23.mp4', 'push-up_24.mp4', 'push-up_25.mp4', 'push-up_26.mp4', 'push-up_27.mp4', 'push-up_28.mp4', 'push-up_29.mp4', 'push-up_3.mp4', 'push-up_30.mp4', 'push-up_31.mp4', 'push-up_32.mp4', 'push-up_33.mp4', 'push-up_34.mp4', 'push-up_35.mp4', 'push-up_36.mp4', 'push-up_38.mp4', 'push-up_39.mp4', 'push-up_4.mp4', 'push-up_41.mp4', 'push-up_42.mp4', 'push-up_44.mp4', 'push-up_45.mp4', 'push-up_46.mp4', 'push-up_47.mp4', 'push-up_48.mp4', 'push-up_49.mp4', 'push-up_5.mp4', 'push-up_50.mp4', 'push-up_51.mp4', 'push-up_52.mp4', 'push-up_53.mp4', 'push-up_6.mp4', 'push-up_7.mp4', 'push-up_8.mp4', 'push-up_9.mp4']

In [9]:
# Split the dataset into training and testing sets
# 80% of the data for training, 20% for testing
# random_state=42 ensures reproducibility
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and Train the XGBoost Classifier
model = XGBClassifier()       # Create an instance of the XGBoost classifier
model.fit(X_train, y_train)   # Train the model on the training data

# Evaluate the Model on the Test Set
y_pred = model.predict(X_test)  # Predict labels for the test data

# Print a classification report including precision, recall, f1-score, and accuracy
print(classification_report(y_test, y_pred, target_names=["Good Form", "Bad Form"]))

              precision    recall  f1-score   support

   Good Form       0.67      0.67      0.67         3
    Bad Form       0.89      0.89      0.89         9

    accuracy                           0.83        12
   macro avg       0.78      0.78      0.78        12
weighted avg       0.83      0.83      0.83        12



In [22]:
# Define the path
final_model_path = r"C:\Users\derek\.cache\kagglehub\datasets\hasyimabdillah\workoutfitness-video\versions\5\push-up\form_classifier_model.pkl"

# Make sure the directory exists
os.makedirs(os.path.dirname(final_model_path), exist_ok=True)

# Save the model
joblib.dump(model, final_model_path)

print(final_model_path)

C:\Users\derek\.cache\kagglehub\datasets\hasyimabdillah\workoutfitness-video\versions\5\push-up\form_classifier_model.pkl


In [23]:
from ultralytics import YOLO
import cv2
import pandas as pd
import numpy as np
import joblib
import os
import time
from datetime import datetime

# Load models
yolo_model = YOLO("yolov8n-pose.pt")
classifier = joblib.load(os.path.join(final_model_path))

cap = cv2.VideoCapture(0)
data = []
frame_num = 0
frame_skip = 1
last_grade_time = time.time()  # Initialize to now, not 0

print("Press 'q' to quit. Press 's' to score your current form. Form automatically assessed every five seconds.")

def grade_form(data, classifier):
    """Score form from collected keypoint data. Returns label string or None if insufficient data."""
    if len(data) == 0:
        return None
    df = pd.DataFrame(data)
    # Need at least 2 frames worth of data for std() to be meaningful
    if df['frame'].nunique() < 2:
        return None
    grouped = df.groupby(['frame', 'keypoint_id'])[['x', 'y']].mean().unstack()
    features = grouped.mean().tolist() + grouped.std().fillna(0).tolist()
    X = np.array([features])
    prediction = classifier.predict(X)
    return "Great form!" if prediction[0] == 0 else "Form could use some work"

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    if frame_num % frame_skip == 0:
        results = yolo_model(frame, verbose=False)
        annotated_frame = results[0].plot()

        # Grade form every 5 seconds
        now = time.time()
        if now - last_grade_time >= 5:
            label = grade_form(data, classifier)
            if label:
                print(f"Form Assessment: {label}")
            else:
                print("Not enough data yet for form assessment.")
            last_grade_time = now  # Reset the timer

        keypoints = results[0].keypoints
        if keypoints is not None and len(keypoints.xy) > 0:
            for i, kp in enumerate(keypoints.xy[0]):
                x, y = kp[0].item(), kp[1].item()
                conf = keypoints.conf[0][i].item()
                data.append({
                    'frame': frame_num,
                    'keypoint_id': i,
                    'x': x, 'y': y,
                    'confidence': conf
                })

        cv2.imshow("Exercise Form Analyzer - Press Q to quit", annotated_frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('s'):
        label = grade_form(data, classifier)
        if label:
            print(f"Form Assessment: {label}")
        else:
            print("Not enough data yet for form assessment.")

    frame_num += 1

cap.release()
cv2.destroyAllWindows()

Press 'q' to quit. Press 's' to score your current form. Form automatically assessed every five seconds.
Form Assessment: Great form!
Form Assessment: Great form!
